# Milestone 4 - System Prototype

**Project:** NLP-assisted job opportunity matching for MSBA international students

**Team GitHub notebook URL:** https://github.com/Kongbai815/job-matching-nlp/blob/main/MSBA_Job_Matching_Milestone4_System_Prototype.ipynb

This notebook demonstrates a working end-to-end prototype: real advisor input -> retrieval over real postings -> grounded recommendation output -> evaluation against the Milestone 2 baseline.

## 1. Prototype Architecture

The system follows a retrieval-augmented pattern. It does not freely generate unsupported claims. Instead, it retrieves similar real job postings, predicts a triage label using retrieved evidence plus a transparent rubric, and generates a template answer that cites the retrieved posting fields.

In [1]:
from pathlib import Path
import json
import pandas as pd

DATA_PATH = Path('data/data_jobs_msba_project_sample_100k.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data_jobs_msba_project_sample_100k.csv')
RESULTS_PATH = Path('milestone4_prototype_results.json')
DEMO_PATH = Path('milestone4_demo_output.json')
df = pd.read_csv(DATA_PATH)
results = json.loads(RESULTS_PATH.read_text())
demo = json.loads(DEMO_PATH.read_text())
print('Loaded project rows:', len(df))
print('Train rows:', (df['split'] == 'train').sum())
print('Validation rows:', (df['split'] == 'validation').sum())


Loaded project rows: 100000
Train rows: 80000
Validation rows: 20000


## 2. Real Input

In [2]:
print(json.dumps(demo['input'], indent=2))


{
  "user": "Graduate business career advisor",
  "query": "Find entry-level US analytics or business analyst opportunities for MSBA international students with SQL, Python, Excel, Tableau, or Power BI skills. Flag cases where authorization evidence is missing.",
  "student_profile": {
    "program": "MSBA",
    "career_goal": "data analyst, business analyst, BI analyst, or operations analytics",
    "skills": [
      "SQL",
      "Python",
      "Excel",
      "Tableau",
      "Power BI"
    ],
    "location_preference": "United States, Bay Area or remote preferred",
    "authorization_need": "CPT/OPT evidence must be reviewed by an advisor"
  }
}


## 3. Grounded System Output

In [3]:
print(demo['system_output']['headline'])
print('\nRecommended action:', demo['system_output']['recommended_action'])
print('\nGrounding caveat:', demo['system_output']['grounding_caveat'])
print('\nTop retrieved evidence:')
for item in demo['system_output']['retrieved_evidence'][:5]:
    print('-', item['posting_id'], item['source_title'], '|', item['company'], '|', item['grounded_fit_label'])
    for ev in item['evidence']:
        print('  evidence:', ev)


Review 3 of the retrieved postings first; they match analytics role-fit signals.

Recommended action: Advisor review before forwarding to students

Grounding caveat: Do not infer CPT, OPT, or sponsorship from this dataset. The public source does not include reliable authorization text, so every recommendation keeps authorization as a human-review item.

Top retrieved evidence:
1. Entry level / Busines Data Analyst (Remote) at Lumos Stratgy (San Francisco, CA) - high_fit
2. Entry Level Data Analyst at SkiHomeRealty (Anywhere) - high_fit
3. Entry Level Business Analyst/Data Analyst at Alipro (Tysons, VA) - high_fit
4. Entry Level Data Analyst - US Army (13J) at United States Army (Long Beach, CA) - unclear
5. Business Analyst in BI Analytics Guild at Swedbank (Anywhere) - unclear


## 4. Evaluation Against Baseline

| System | Eval rows | Accuracy | Macro F1 |
| --- | ---: | ---: | ---: |
| Milestone 2 published baseline | 20,000 | 0.793 | 0.785 |
| Same-subset hashed centroid baseline | 4,000 | 0.796 | 0.791 |
| Milestone 4 retrieval + grounded generation prototype | 4,000 | 0.800 | 0.801 |

Grounding metrics: retrieval success rate = **100.0%**, support hit@5 = **90.0%**.

In [4]:
metrics = results['prototype_evaluation']['metrics']
print('Prototype accuracy:', round(metrics['accuracy'], 4))
print('Prototype macro F1:', round(metrics['macro_f1'], 4))
print('Retrieval success rate:', round(results['prototype_evaluation']['grounding']['retrieval_success_rate'], 4))
print('Support hit@5:', round(results['prototype_evaluation']['grounding']['support_hit_at_5'], 4))


Prototype accuracy: 0.8005
Prototype macro F1: 0.8009
Retrieval success rate: 1.0000
Support hit@5: 0.8995


## 5. Risk and Grounding Analysis

The largest hallucination risk is work-authorization language. The public source data does not contain reliable CPT, OPT, or sponsorship text, so the prototype never claims that a role is visa-friendly. It only says that authorization evidence is missing and routes that part to human review. The second risk is weak-label overconfidence: the evaluation labels are rule-derived, not final advisor labels. For governance, the prototype shows retrieved posting evidence and keeps the advisor in the loop before forwarding opportunities.